# Computational Physics — Course Walkthrough

This notebook documents the numerical methods and formulas used across the Fortran and Python examples in this folder, with concise, runnable Python demonstrations for key algorithms.

Files referenced (examples): `jacobian.f90`, `solve_lin_system.f90`, `mat_inv.f90`, `matrix_multi.f90`, `simpsons.f90`, `gauss_quad.f90`, `series_1.f90`, `assgn_3.f90`, `sho.f90`, `solve_diffu.f90`, `solve_laplace.f90`, `assgn_2_2.f90`, and plotting scripts `plot_series.py`, `plot_assgn_3.py`, `sho_plot.py`, `plot_E.py`.

## Table of contents
- Numerical differentiation and Jacobian
- Linear algebra: solving systems, inverse, LU / LAPACK calls
- Matrix multiplication
- Numerical integration: Simpson and Gauss quadrature
- Series summation and precision effects
- ODE solvers: Euler and RK2 examples
- PDE examples: Diffusion (explicit) and Laplace (relaxation)
- Root finding (Newton) and quantum well example
- Quick plotting notes and how to run Fortran examples

**How to compile and run Fortran examples**

Typical commands (from this directory):

```bash
gfortran jacobian.f90 -o jacobian && ./jacobian
gfortran solve_diffu.f90 -o diff && ./diff
gfortran solve_laplace.f90 -o laplace && ./laplace
```

Files that produce data files (e.g., `phi.dat`, `E.dat`, `result_Euler.out`) are used by the Python plotting scripts in this folder.

## Numerical differentiation and Jacobian

Concepts: finite-difference approximations for derivatives. Common formulas used in the Fortran code: central difference (first derivative) and second-order central-difference for second derivatives.

Central difference (first derivative):
$$f'(x) \approx \frac{f(x+h)-f(x-h)}{2h}$$

Second derivative (central, used implicitly by 3-point stencils):
$$f''(x) \approx \frac{f(x+h)-2f(x)+f(x-h)}{h^2}$$

The files `jacobian.f90`, `jacobian_1.f90`, and `jacobian_2.f90` compute a 3x3 Jacobian for a nonlinear system using finite differences. `jacobian_2.f90` shows an explicit central-difference implementation and prints a 3x3 matrix.

In [ ]:
# Python demonstration: compute Jacobian of a vector function via central differences
import numpy as np

def jacobian_central(fun, x, h=1e-6):
    x = np.asarray(x, dtype=float)
    n = x.size
    f0 = np.asarray(fun(x))
    m = f0.size
    J = np.zeros((m, n), dtype=float)
    for j in range(n):
        xph = x.copy(); xmh = x.copy()
        xph[j] += h
        xmh[j] -= h
        J[:, j] = (fun(xph) - fun(xmh)) / (2*h)
    return J

def F(x):
    x1,x2,x3 = x
    return np.array([x1+x2+x3-6, x1*x2 + x2*x3 + x3*x1 - 11, x1*x2*x3 - 36])

x0 = np.array([2.0, 0.5, 1.0])
J = jacobian_central(F, x0)
J

## Linear algebra: solving systems and matrix inverse

The Fortran `solve_lin_system.f90` calls `dgesv` (LAPACK) to solve Ax=b. `mat_inv.f90` uses `dgetrf` + `dgetri` to compute an inverse. For teaching, `numpy.linalg.solve` and `numpy.linalg.inv` provide the same interfaces in Python.

In [ ]:
import numpy as np
A = np.array([[2.0, 3.0],[1.0,1.0]])
b = np.array([5.0,6.0])
x = np.linalg.solve(A,b)
A_inv = np.linalg.inv(A)
x, A_inv

## Matrix multiplication

Several Fortran examples show explicit triple-loop matrix multiplication (`matrix_multi.f90`, `Matrix-multi.f90`). This is O(n^3). In Python use `numpy.matmul` or the `@` operator for performance.

In [ ]:
# small demo
A = np.arange(1,7).reshape(3,2).astype(float)
B = np.arange(7,15).reshape(2,4).astype(float)
C = A @ B
C

## Numerical integration: Simpson's rule and Gauss quadrature

Fortran files `simpsons*.f90` implement Simpson's rule; `gauss_quad.f90` reads weights & points and applies Gaussian quadrature. Key formula (Simpson on even number of intervals):
$$\int_a^b f(x) \; dx \approx \frac{h}{3} \left(f_0 + f_n + 4\sum_{\text{odd}} f_i + 2\sum_{\text{even}} f_i\right)$$

In [ ]:
# Simpson's rule implementation
def simpson(f, a, b, n):
    if n % 2 == 1:
        raise ValueError('n must be even')
    h = (b-a)/n
    x = np.linspace(a,b,n+1)
    y = f(x)
    return h/3*(y[0] + y[-1] + 4*y[1:-1:2].sum() + 2*y[2:-1:2].sum())

# Gaussian quadrature via numpy.polynomial.legendre
from numpy.polynomial.legendre import leggauss
def gauss_quad(f, a, b, n=10):
    x,w = leggauss(n)
    # map from [-1,1] to [a,b]
    xp = 0.5*(b-a)*x + 0.5*(b+a)
    return 0.5*(b-a)*np.dot(w, f(xp))

f = lambda x: x**2
simpson(f,1,3,100) , gauss_quad(f,1,3,10)

## Series summation and precision effects

The Fortran assignments (`assgn_1_1.f90`, `assgn_1.f90`, `series_1.f90`) explore summing the harmonic series in ascending vs descending order and the impact of single vs double precision. Summation order affects rounding error. The recommended exercise is to compare `float32` vs `float64` summation and ascending vs descending accumulation.

In [ ]:
# Demonstration: summation order and float32 vs float64
def sum_up(N, dtype=np.float64):
    s = dtype(0)
    for k in range(1, N+1):
        s += dtype(1)/dtype(k)
    return s

def sum_down(N, dtype=np.float64):
    s = dtype(0)
    for k in range(N, 0, -1):
        s += dtype(1)/dtype(k)
    return s

N = 100000
sum_up(N, np.float32), sum_down(N, np.float32), sum_up(N, np.float64), sum_down(N, np.float64)

## ODE solvers: Euler and RK2

The Fortran `assgn_3.f90`, `shm_rk2.f90`, and `sho.f90` implement explicit Euler and a second-order Runge–Kutta (midpoint/RK2) integrator. RK2 (midpoint) step for y' = f(t,y):
k1 = f(t, y)
k2 = f(t + dt/2, y + dt/2 * k1)
y_{n+1} = y_n + dt * k2

`assgn_3.f90` implements RK2 for a simple Lotka–Volterra-like system; `sho.f90` uses Euler for a simple harmonic oscillator.

In [ ]:
# RK2 demo: Lotka-Volterra-like system from assgn_3.f90
import matplotlib.pyplot as plt
def rk2_step(f, t, y, dt):
    k1 = f(t, y)
    k2 = f(t + 0.5*dt, y + 0.5*dt*k1)
    return y + dt*k2

def LV(t, y):
    x,yv = y
    dx = 1.2*x - 0.6*x*yv
    dy = -0.8*yv + 0.3*x*yv
    return np.array([dx, dy])

dt = 0.01
t = 0.0
y = np.array([2.0, 1.0])
ts, xs, ys = [t], [y[0]], [y[1]]
for i in range(3000):
    y = rk2_step(LV, t, y, dt)
    t += dt
    ts.append(t); xs.append(y[0]); ys.append(y[1])

plt.plot(ts, xs, label='x'); plt.plot(ts, ys, label='y'); plt.legend(); plt.show()

## PDE examples: Diffusion (explicit) and Laplace (relaxation)

`solve_diffu.f90` demonstrates an explicit forward-time centered-space (FTCS) scheme for diffusion with a source: update phi += dt * RHS where RHS contains the second spatial derivative (D/dx^2 * (phi_{i+1}-2phi_i+phi_{i-1})) and a nonlinear source term. Stability constraint for explicit diffusion:
$$dt \le \frac{dx^2}{2D}$$ (practical `lambda = D*dt/dx^2` should be <= 0.5 for stability in 1D explicit schemes).

`solve_laplace.f90` solves 2D Laplace via Jacobi iteration:
phi_{i,j} = 0.25*(phi_{i+1,j} + phi_{i-1,j} + phi_{i,j+1} + phi_{i,j-1})

In [ ]:
# Simple 1D diffusion explicit update demo (toy)
def diffusion_step(phi, D, dx, dt):
    phi_new = phi.copy()
    fct = D/dx**2
    for i in range(1, len(phi)-1):
        phi_new[i] = phi[i] + dt*(fct*(phi[i+1]-2*phi[i]+phi[i-1]))
    return phi_new

# 2D Jacobi relaxation for Laplace
def jacobi_step(phi):
    phi2 = phi.copy()
    nx,ny = phi.shape
    for i in range(1,nx-1):
        for j in range(1,ny-1):
            phi2[i,j] = 0.25*(phi[i+1,j]+phi[i-1,j]+phi[i,j+1]+phi[i,j-1])
    return phi2

# tiny sanity run omitted for brevity
print('Defined diffusion and jacobi helper functions')

## Root finding and the quantum well example

`assgn_2_2.f90` and `Q2.f90` use Newton's method to find energy eigenvalues for a finite quantum well by applying root-finding to transcendental matching conditions for symmetric and anti-symmetric wavefunctions. Newton iteration: x_{n+1} = x_n - f(x_n)/f'(x_n). Always supply a good initial guess and check convergence.

In [ ]:
# Simple Newton solver demo (1D)
def newton(f, df, x0, tol=1e-8, maxit=50):
    x = x0
    for i in range(maxit):
        fx = f(x)
        if abs(fx) < tol: return x
        x = x - fx/df(x)
    raise RuntimeError('No convergence')

# Example: solve sqrt(100-x)*tan(sqrt(100-x)) - sqrt(x) = 0 (as in Q2.f90)
import math
def f(E): return math.sqrt(100.0-E)*math.tan(math.sqrt(100.0-E)) - math.sqrt(E)
def df(E): return -0.5*(1.0/math.sqrt(E) + math.tan(math.sqrt(100.0-E))/math.sqrt(100.0-E) + math.tan(math.sqrt(100.0-E))**2 + 1.0)
# newton(f, df, 4.0)  # numerical run omitted to keep the cell quick
print('Newton helper defined')

## Plotting and visualization

Plotting scripts included: `plot_assgn_3.py` (phase/solutions), `sho_plot.py` (energy vs time), `plot_series.py` and `plot_sumseries.py` for summation experiments, `plot_E.py` for electric field visualization from `E.dat`. Use `python plot_assgn_3.py` etc.

## Next steps and exercises
- Run the Fortran programs, inspect produced `.dat` files, and use the provided Python plotting scripts.
- Convert one Fortran example to a Python implementation and compare results and performance.
- Add unit tests for small numerical routines (Jacobian, RK2) to verify convergence orders.

If you'd like, I can now: (A) run a few short Python demos here to verify behavior, (B) run/compile selected Fortran files and capture outputs, or (C) expand any notebook section into a deeper, interactive teaching cell set. Which would you prefer?

## Convergence tests and hands-on exercises

This section adds runnable convergence tests you can use to verify the expected error scaling (order) of finite-difference derivatives, Simpson integration, and the RK2 time integrator. Run each cell and observe error vs. step size plots / tables.

### Derivative convergence (central difference)

We test f(x)=sin(x) at x=1.0; exact derivative is cos(1.0). Central difference should be O(h^2).

In [ ]:
import numpy as np
def central_derivative(f, x, h):
    return (f(x+h)-f(x-h))/(2*h)
f = np.sin
x0 = 1.0
hs = np.logspace(-8, -1, 8)
errs = []
for h in hs:
    num = central_derivative(f, x0, h)
    err = abs(num - np.cos(x0))
    errs.append(err)
list(zip(hs, errs))

### Simpson convergence test

Integrate f(x)=sin(x) from 0..pi (exact = 2). Simpson's rule is O(h^4) overall when composite with even n; observe error scaling.

In [ ]:
import numpy as np
def simpson_comp(f,a,b,n):
    if n%2==1: n+=1
    h=(b-a)/n
    x=np.linspace(a,b,n+1)
    y=f(x)
    return h/3*(y[0]+y[-1]+4*y[1:-1:2].sum()+2*y[2:-1:2].sum())
f=lambda x: np.sin(x)
ns = [10,20,40,80,160,320]
errs=[]
for n in ns:
    val=simpson_comp(f,0,np.pi,n)
    errs.append(abs(val-2.0))
list(zip(ns, errs))

### RK2 temporal convergence and energy test

Solve dy/dt = -y with y(0)=1. Exact y=exp(-t). RK2 is second order in dt. We'll measure error at t=1 for decreasing dt.

In [ ]:
import numpy as np
def rk2_integrate(f, y0, t0, t1, dt):
    t=t0; y=y0
    while t < t1 - 1e-12:
        h=min(dt, t1-t)
        k1=f(t,y)
        k2=f(t+0.5*h, y+0.5*h*k1)
        y = y + h * k2
        t += h
    return y
f=lambda t,y: -y
dts = [0.1, 0.05, 0.025, 0.0125, 0.00625]
errs=[]
for dt in dts:
    ynum = rk2_integrate(f, 1.0, 0.0, 1.0, dt)
    err = abs(ynum - np.exp(-1.0))
    errs.append(err)
list(zip(dts, errs))

### Jacobian: numeric vs analytic check

Compare the central-difference Jacobian to an analytic Jacobian for a small vector function to ensure correctness of the numerical routine.

In [ ]:
import numpy as np
def Fvec(x):
    x1,x2,x3 = x
    return np.array([x1+x2+x3-6, x1*x2 + x2*x3 + x3*x1 - 11, x1*x2*x3 - 36])
def J_analytic(x):
    x1,x2,x3 = x
    return np.array([[1,1,1],[x2+x3, x1+x3, x1+x2],[x2*x3, x1*x3, x1*x2]])
x0 = np.array([2.0, 3.0, 6.0])
Jnum = jacobian_central(Fvec, x0, h=1e-6)
Jana = J_analytic(x0)
np.set_printoptions(precision=8)
Jana, Jnum, np.abs(Jana-Jnum)

### Fortran compile/run helper

Below are safe example commands to compile selected Fortran examples from this directory. Run them in a terminal (not inside the Python kernel) to produce `.dat` output files consumed by the Python plotting scripts.

```bash
gfortran jacobian.f90 -o jacobian && ./jacobian
gfortran solve_diffu.f90 -o diff && ./diff
gfortran solve_laplace.f90 -o laplace && ./laplace
```

If you want, I can run quick Python validations here (safe, read-only) or attempt to compile Fortran (requires using the workspace terminal). Which do you prefer now?